<a href="https://colab.research.google.com/github/vrani2208/Automation-loops/blob/main/ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

RAW_CSV = """name,online_order,rate,cost_for_two,cuisines
Spice Garden,Yes,4.1/5,500,"North Indian, Chinese"
Biryani House,No,3.8/5,"1,200","Biryani, North Indian, Mughlai"
South Bites,Yes,NEW,300,South Indian
The Grill Room,Yes,4.5/5,800,"Continental, Italian, Mediterranean"
Street Eats,No,-,150,Street Food
Coastal Curry,No,3.5/5,350,"Seafood, South Indian"
Cafe Bliss,Yes,4.0/5,450,"Cafe, Desserts"
Delhi Dhaba,No,3.9/5,"1,000","North Indian, Mughlai, Chinese, Rajasthani"
Sushi Spot,Yes,4.2/5,600,"Japanese, Asian"
Tandoor Tales,Yes,4.4/5,700,"North Indian, Tandoor, Kebabs"
"""

def load_and_clean(raw_csv):
    df = pd.read_csv(StringIO(raw_csv))

    # Clean rate -> rating
    df['rating'] = df['rate'].str.replace('/5', '', regex=False)
    df['rating'] = df['rating'].replace(['NEW', '-'], np.nan)
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

    # Clean cost_for_two
    df['cost_for_two'] = df['cost_for_two'].astype(str).str.replace(',', '', regex=False)
    df['cost_for_two'] = pd.to_numeric(df['cost_for_two'], errors='coerce')

    # Drop rows with missing rating
    df = df.dropna(subset=['rating']).copy()

    # Feature engineering: cuisine_count
    df['cuisine_count'] = df['cuisines'].str.split(',').str.len()

    return df

def analyse(df):
    print("Shape:", df.shape)
    print("\nRating stats:")
    print(df['rating'].describe())
    print("\nCost for two stats:")
    print(df['cost_for_two'].describe())
    print("\nMean rating by online_order:")
    print(df.groupby('online_order')['rating'].mean())
    print("\nCuisine count distribution:")
    print(df['cuisine_count'].value_counts())

def plot_rating_histogram(df):
    sns.histplot(df['rating'], bins=10)
    plt.title('Rating Distribution')
    plt.xlabel('Rating')
    plt.ylabel('Count')
    plt.close()

if __name__ == "__main__":
    df = load_and_clean(RAW_CSV)
    analyse(df)
    plot_rating_histogram(df)


Shape: (8, 7)

Rating stats:
count    8.000000
mean     4.050000
std      0.325137
min      3.500000
25%      3.875000
50%      4.050000
75%      4.250000
max      4.500000
Name: rating, dtype: float64

Cost for two stats:
count       8.000000
mean      700.000000
std       289.087233
min       350.000000
25%       487.500000
50%       650.000000
75%       850.000000
max      1200.000000
Name: cost_for_two, dtype: float64

Mean rating by online_order:
online_order
No     3.733333
Yes    4.240000
Name: rating, dtype: float64

Cuisine count distribution:
cuisine_count
2    4
3    3
4    1
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')
# Load the CarDekho dataset
df = pd.read_csv('cardekho_dataset.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (15411, 14)
Columns: ['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']


,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [ ]:
df = df.drop(columns=['Unnamed: 0', 'car_name', 'model'])
print(f"Shape after dropping: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Shape after dropping: (15411, 11)
Remaining columns: ['brand', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']


In [ ]:
df


,brand,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...
15406,Hyundai,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,Maruti,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,Skoda,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,Mahindra,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)



test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)



print("=" * 55)
print(f"{'Metric':<10} {'Train':>15} {'Test':>15}")
print("=" * 55)
print(f"{'MAE':<10} {'₹{:,.0f}'.format(train_mae):>15} {'₹{:,.0f}'.format(test_mae):>15}")
print(f"{'RMSE':<10} {'₹{:,.0f}'.format(train_rmse):>15} {'₹{:,.0f}'.format(test_rmse):>15}")
print(f"{'R²':<10} {train_r2:>15.4f} {test_r2:>15.4f}")
print("=" * 55)

print(f"\nInterpretation:")
print(f"  • On average, predictions are off by ₹{test_mae:,.0f} on unseen data")
print(f"  • The model explains {test_r2*100:.1f}% of the variance in car prices")
